# 28 · Pig counting robustness and statistical uncertainty
**Project:** FaroPigSeg — First scientific article  
**Runtime:** **CPU** (GPU is not required).  
**Input:** `CERDOS_27_Resultados.zip` (experiment 27).  
**Outputs:** Excel tables, publication figures in English (600 DPI PNG, SVG and PDF), provenance JSON, downloadable ZIP.  

**Scientific interpretation:** The test set was previously inspected in experiments 23–25. Results are *descriptive and exploratory*, not independent confirmation. Acquisition-date resampling has **only five clusters**: cluster-bootstrap intervals are unstable and must not be represented as definitive inferential evidence. The three seeds share the same images and are not independent test samples. No thresholds are retuned here.

In [ ]:
# 1 · Setup — CPU
import sys, subprocess, importlib.util
for package in ('pandas','openpyxl','matplotlib','numpy'):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',package])
from pathlib import Path
import os, io, json, zipfile, hashlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from datetime import datetime, timezone
print('Python:',sys.version.split()[0], '| CPU analysis; GPU not required')

In [ ]:
# 2 · Locate existing experiment 27 output in your Drive; upload only if necessary
try:
    from google.colab import drive, files
    drive.mount('/content/drive')
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
PROJECT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖') if IN_COLAB else Path('/mnt/data')
SEARCH_ROOTS=[PROJECT/'03_Resultados', PROJECT/'05_Exportaciones_ZIP', PROJECT, Path('/content'),Path('/mnt/data')]
candidates=[]
for folder in SEARCH_ROOTS:
    if folder.exists():
        candidates.extend(folder.rglob('CERDOS_27_Resultados*.zip'))
candidates=sorted(set(candidates), key=lambda x:x.stat().st_mtime,reverse=True)
if candidates:
    INPUT_ZIP=candidates[0]
    print('Source:',INPUT_ZIP)
elif IN_COLAB:
    print('Upload CERDOS_27_Resultados.zip from experiment 27:')
    uploaded=files.upload()
    matches=[Path('/content')/k for k in uploaded if '27' in k and k.lower().endswith('.zip')]
    if not matches: raise FileNotFoundError('Upload CERDOS_27_Resultados.zip (experiment 27).')
    INPUT_ZIP=matches[0]
else:
    raise FileNotFoundError('CERDOS_27_Resultados.zip not found')
assert INPUT_ZIP.is_file()

In [ ]:
# 3 · Strict schema and paired-data integrity checks
with zipfile.ZipFile(INPUT_ZIP) as archive:
    names=archive.namelist()
    excel_names=[s for s in names if s.endswith('.xlsx') and not s.startswith('__MACOSX')]
    if len(excel_names)!=1: raise ValueError(f'Expected exactly one Excel workbook, got {excel_names}')
    source_bytes=archive.read(excel_names[0])
    xls=pd.ExcelFile(io.BytesIO(source_bytes))
    required={'Per_Image','Paired_Image_Deltas','Locked_Thresholds','Global_Comparison'}
    if not required.issubset(set(xls.sheet_names)):raise ValueError(f'Missing sheets {required-set(xls.sheet_names)}')
    df=pd.read_excel(xls,'Per_Image')
    locked=pd.read_excel(xls,'Locked_Thresholds')
    paired_previous=pd.read_excel(xls,'Paired_Image_Deltas')
needed={'semilla','image_id','fecha','imagen','gt_conteo','strategy','threshold','predicted_count','absolute_error','signed_error'}
if not needed.issubset(df.columns):raise ValueError(f'Missing columns {needed-set(df.columns)}')
strategies={'Original (0.25)','Validation-calibrated'}
if set(df.strategy.astype(str))!=strategies:raise ValueError('Unexpected strategy labels')
if df[['semilla','image_id','strategy']].duplicated().any():raise ValueError('Duplicate seed/image/strategy')
if df[list(needed)].isna().any().any():raise ValueError('Missing input values')
if len(locked)!=len(locked.seed.unique()):raise ValueError('Duplicate frozen seed thresholds')
seeds=sorted(df.semilla.unique().tolist());dates=sorted(df.fecha.astype(str).unique().tolist())
if len(seeds)!=3:raise ValueError(f'Expected 3 seeds, found {seeds}')
if len(dates)!=5:raise ValueError(f'Expected 5 acquisition dates, found {dates}')
if not np.allclose(df.absolute_error,(df.predicted_count-df.gt_conteo).abs()):raise ValueError('Absolute errors inconsistent')
if not np.allclose(df.signed_error,df.predicted_count-df.gt_conteo):raise ValueError('Signed errors inconsistent')
for seed in seeds:
    part=df[df.semilla==seed]
    base=part[part.strategy=='Original (0.25)'].sort_values('image_id')
    new=part[part.strategy=='Validation-calibrated'].sort_values('image_id')
    if not np.array_equal(base.image_id.to_numpy(),new.image_id.to_numpy()):raise ValueError('Unpaired images')
    if not np.array_equal(base.gt_conteo.to_numpy(),new.gt_conteo.to_numpy()):raise ValueError('Ground truth differs')
    threshold=float(locked.loc[locked.seed==seed,'threshold'].iloc[0])
    if not np.allclose(new.threshold,threshold):raise ValueError(f'Frozen threshold mismatch for {seed}')
    if not np.allclose(base.threshold,0.25):raise ValueError('Original threshold mismatch')
print(f'Integrity PASS | {len(seeds)} seeds | {len(dates)} dates | {len(df)//(2*len(seeds))} images/seed')
print('Frozen thresholds:',dict(zip(locked.seed,locked.threshold)))

In [ ]:
# 4 · Paired differences: negative values mean lower absolute count error
pairs=(df.pivot(index=['semilla','image_id','fecha','imagen','gt_conteo'],columns='strategy',values='absolute_error').reset_index())
pairs['difference_cal_minus_orig']=pairs['Validation-calibrated']-pairs['Original (0.25)']
pairs['improved']=pairs.difference_cal_minus_orig<0
pairs['worsened']=pairs.difference_cal_minus_orig>0
pairs['unchanged']=pairs.difference_cal_minus_orig==0
old=paired_previous.sort_values(['semilla','image_id']).reset_index(drop=True)
new=pairs.sort_values(['semilla','image_id']).reset_index(drop=True)
if len(old)!=len(new) or not np.allclose(old['MAE_difference_calibrated_minus_original'],new.difference_cal_minus_orig):
    raise ValueError('Paired differences do not reproduce experiment 27')
summary=pairs.groupby('semilla').agg(n_images=('image_id','size'),mae_original=('Original (0.25)','mean'),mae_calibrated=('Validation-calibrated','mean'),mean_paired_difference=('difference_cal_minus_orig','mean'),median_difference=('difference_cal_minus_orig','median'),improved_pct=('improved','mean'),worsened_pct=('worsened','mean'),unchanged_pct=('unchanged','mean')).reset_index()
for c in ('improved_pct','worsened_pct','unchanged_pct'):summary[c]*=100
summary['relative_MAE_reduction_pct']=100*(summary.mae_original-summary.mae_calibrated)/summary.mae_original
by_date=pairs.groupby(['semilla','fecha']).agg(n_images=('image_id','size'),mae_original=('Original (0.25)','mean'),mae_calibrated=('Validation-calibrated','mean'),mean_paired_difference=('difference_cal_minus_orig','mean')).reset_index()
print(summary.round(3).to_string(index=False))

In [ ]:
# 5 · Two uncertainty analyses, clearly distinguished
# A) image-resampling bootstrap: DESCRIPTIVE ONLY; images are not independent within acquisition dates.
# B) cluster bootstrap by acquisition date: only FIVE clusters, thus highly unstable.
# Same image sampling is reused across seeds within each bootstrap draw (paired structure preserved).
RNG=np.random.default_rng(20261019);B=3000
image_ids=np.array(sorted(pairs.image_id.unique()))
wide=pairs.pivot(index='image_id',columns='semilla',values='difference_cal_minus_orig').loc[image_ids]
image_draws=RNG.integers(0,len(image_ids),size=(B,len(image_ids)))
img_boot=wide.to_numpy()[image_draws].mean(axis=1)
# Pool ALL images from resampled dates, respecting observed sample counts per date.
date_list=sorted(pairs.fecha.astype(str).unique())
date_vectors={d:wide.loc[pairs.loc[pairs.semilla==seeds[0]].set_index('image_id').query('fecha == @d').index].to_numpy() for d in date_list}
cluster_draws=RNG.integers(0,len(date_list),size=(B,len(date_list)))
cluster_boot=np.empty((B,len(seeds)))
for b,row in enumerate(cluster_draws):
    counts=np.bincount(row,minlength=len(date_list))
    numerator=sum(counts[j]*date_vectors[date_list[j]].sum(axis=0) for j in range(len(date_list)))
    denom=sum(counts[j]*len(date_vectors[date_list[j]]) for j in range(len(date_list)))
    cluster_boot[b]=numerator/denom
records=[]
for k,seed in enumerate(seeds):
    records.append({'semilla':seed,'point_mean_difference':wide[seed].mean(),
        'image_bootstrap_CI_low':np.quantile(img_boot[:,k],.025),'image_bootstrap_CI_high':np.quantile(img_boot[:,k],.975),
        'date_cluster_bootstrap_CI_low':np.quantile(cluster_boot[:,k],.025),'date_cluster_bootstrap_CI_high':np.quantile(cluster_boot[:,k],.975),
        'n_date_clusters':len(date_list),'n_bootstrap':B,
        'interpretation':'Exploratory; five date clusters are too few for reliable inferential coverage'})
intervals=pd.DataFrame(records)
print(intervals.round(3).to_string(index=False))

In [ ]:
# 6 · Leave-one-date-out robustness (sensitivity, NOT external validation)
loo=[]
for d in date_list:
    for seed in seeds:
        x=pairs[(pairs.semilla==seed)&(pairs.fecha.astype(str)!=d)]
        loo.append({'excluded_date':d,'semilla':seed,'remaining_images':len(x),
                    'mean_paired_difference':x.difference_cal_minus_orig.mean(),
                    'MAE_original':x['Original (0.25)'].mean(),'MAE_calibrated':x['Validation-calibrated'].mean()})
loo=pd.DataFrame(loo)
# Cross-seed agreement per image (dependent model predictions)
agreement=pairs.pivot(index=['image_id','fecha'],columns='semilla',values='difference_cal_minus_orig').reset_index()
agreement['all_seeds_improved']=(agreement[seeds]<0).all(axis=1)
agreement['all_seeds_worsened']=(agreement[seeds]>0).all(axis=1)
agreement['all_seeds_nonworse']=(agreement[seeds]<=0).all(axis=1)
print('Images improved for all seeds:',int(agreement.all_seeds_improved.sum()),'/',len(agreement))

In [ ]:
# 7 · Journal-style publication graphics (all English)
OUT=PROJECT/'03_Resultados'/'CERDOS_28_Robustez_Estadistica_Conteo' if IN_COLAB else Path('/mnt/data/CERDOS_28_Robustez_Estadistica_Conteo')
FIG=OUT/'Figures';FIG.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.labelsize':11,'axes.titlesize':12,'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':120,'savefig.dpi':600,'pdf.fonttype':42,'svg.fonttype':'none'})
colors=['#0072B2','#D55E00','#009E73']
def savefig(name):
    plt.tight_layout()
    for ext in ('png','pdf','svg'):plt.savefig(FIG/f'{name}.{ext}',dpi=600,bbox_inches='tight')
    plt.show();plt.close()
fig,ax=plt.subplots(figsize=(7.2,4.4))
x=np.arange(len(seeds));w=.34
ax.bar(x-w/2,summary.mae_original,w,label='Original threshold (0.25)',color='#808080')
ax.bar(x+w/2,summary.mae_calibrated,w,label='Validation-calibrated threshold',color=colors[0])
ax.set_xticks(x,[f'Seed {s}' for s in seeds]);ax.set_ylabel('Mean absolute counting error (pigs/image)')
ax.set_title('Pig counting error before and after calibration');ax.legend(frameon=False);ax.set_ylim(0,4)
savefig('Fig28_01_Paired_MAE_Comparison')
fig,ax=plt.subplots(figsize=(7.2,4.4))
for i,seed in enumerate(seeds):
    g=by_date[by_date.semilla==seed]
    ax.plot(np.arange(len(g)),g.mean_paired_difference,marker='o',color=colors[i],label=f'Seed {seed}')
ax.axhline(0,color='black',lw=1,ls='--');ax.set_xticks(np.arange(len(date_list)),[d[5:] for d in date_list],rotation=30)
ax.set_ylabel('Paired change in absolute error (pigs/image)');ax.set_xlabel('Acquisition date (2023)')
ax.set_title('Counting error change across acquisition dates');ax.legend(frameon=False)
savefig('Fig28_02_Date_Paired_Error_Change')
fig,ax=plt.subplots(figsize=(7.2,4.4))
for i,seed in enumerate(seeds):
    v=intervals[intervals.semilla==seed].iloc[0]
    ax.errorbar(v.point_mean_difference,i,xerr=[[v.point_mean_difference-v.date_cluster_bootstrap_CI_low],[v.date_cluster_bootstrap_CI_high-v.point_mean_difference]],fmt='o',color=colors[i],capsize=4)
ax.axvline(0,color='black',ls='--',lw=1);ax.set_yticks(range(len(seeds)),[f'Seed {s}' for s in seeds]);ax.invert_yaxis()
ax.set_xlabel('Mean paired MAE difference (pigs/image)');ax.set_title('Exploratory date-cluster bootstrap intervals')
ax.text(.01,-.25,'Only five acquisition-date clusters; intervals are unstable',transform=ax.transAxes,fontsize=9)
savefig('Fig28_03_Cluster_Bootstrap_Exploratory')
fig,ax=plt.subplots(figsize=(7.2,4.4))
for i,seed in enumerate(seeds):
    g=loo[loo.semilla==seed];ax.plot(range(len(g)),g.mean_paired_difference,marker='s',color=colors[i],label=f'Seed {seed}')
ax.axhline(0,color='black',ls='--',lw=1);ax.set_xticks(range(len(date_list)),[d[5:] for d in date_list],rotation=30)
ax.set_ylabel('Mean paired MAE difference (pigs/image)');ax.set_xlabel('Excluded acquisition date (2023)')
ax.set_title('Leave-one-date-out sensitivity');ax.legend(frameon=False)
savefig('Fig28_04_Leave_One_Date_Out')

In [ ]:
# 8 · Excel + transparent provenance + ZIP export
book=OUT/'CERDOS_28_Robustez_Estadistica_Conteo.xlsx'
with pd.ExcelWriter(book,engine='openpyxl') as writer:
    for name,data in [('Paired_Summary',summary),('Paired_Per_Image',pairs),('By_Date',by_date),('Bootstrap_Intervals',intervals),('LODO_Sensitivity',loo),('Seed_Agreement',agreement),('Locked_Thresholds',locked)]:data.to_excel(writer,sheet_name=name,index=False)
provenance={'experiment':28,'title':'Pig counting robustness and statistical uncertainty',
            'runtime':'CPU','source_zip_name':INPUT_ZIP.name,'source_zip_SHA256':hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest(),
            'source_workbook_SHA256':hashlib.sha256(source_bytes).hexdigest(),'random_seed':20261019,'bootstrap_repetitions':B,
            'number_of_date_clusters':len(date_list),'dates':date_list,'model_seeds':[int(s) for s in seeds],
            'previous_test_inspection':'Experiments 23–25 inspected test; results descriptive, not independent confirmation',
            'warning':'Five acquisition-date clusters are too few for reliable cluster-bootstrap inference.',
            'timestamp_UTC':datetime.now(timezone.utc).isoformat()}
(OUT/'CERDOS_28_Provenance.json').write_text(json.dumps(provenance,ensure_ascii=False,indent=2),encoding='utf-8')
ZIP=OUT/'CERDOS_28_Resultados.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as archive:
    for f in OUT.rglob('*'):
        if f.is_file() and f!=ZIP:archive.write(f,f.relative_to(OUT))
print('Excel:',book);print('ZIP:',ZIP,'| size:',ZIP.stat().st_size)
if IN_COLAB:
    files.download(str(ZIP))
else:print('Local test complete.')